# VirtualiZarr → Icechunk → Append Copernicus Data

This notebook demonstrates creating an Icechunk store with virtual references to multiple Copernicus Marine Service files, then appending them along the time dimension.

Based on the NDVI CDR append example from NMFS HackDays 2026.

## Workflow

1. Get URLs for multiple Copernicus NetCDF files from S3
2. Open first file as virtual dataset
3. Write virtual references to Icechunk store
4. Loop: open next file, append to Icechunk along time dimension
5. Commit all changes

**Key Point**: No data is downloaded. We only store virtual references to chunks in the original Copernicus S3 files.

In [ ]:
!pip install -qU icechunk virtualizarr copernicusmarine xarray obstore obspec_utils

In [ ]:
import warnings
import shutil
import time
from pathlib import Path

import xarray as xr
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

## Step 1: Get S3 URLs for Multiple Files

We'll get a list of consecutive daily files from Copernicus Marine Service.

In [ ]:
# Get list of files for several consecutive days
# Using July 2024 as an example
!copernicusmarine get \
  --dataset-id cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D \
  --dataset-version 202603 \
  --filter "*202407{01,02,03,04,05}*.nc" \
  --create-file-list copernicus_files_multi.txt

In [ ]:
# Read the S3 URLs
with open('copernicus_files_multi.txt', 'r') as f:
    s3_urls = [line.strip() for line in f if line.strip()]

# Sort to ensure chronological order
s3_urls.sort()

print(f"Found {len(s3_urls)} files")
for url in s3_urls:
    print(f"  {Path(url).name}")

## Step 2: Convert to HTTPS URLs

Convert S3 URLs to HTTPS format for cloudferro endpoint.

In [ ]:
# Copernicus cloudferro endpoint
COPERNICUS_ENDPOINT = "https://s3.waw3-1.cloudferro.com"

def s3_to_https(s3_url, endpoint=COPERNICUS_ENDPOINT):
    """Convert s3://bucket/path to https://endpoint/bucket/path"""
    if s3_url.startswith('s3://'):
        path = s3_url[5:]  # Remove 's3://'
        return f"{endpoint}/{path}"
    return s3_url

https_urls = [s3_to_https(url) for url in s3_urls]
print(f"Converted {len(https_urls)} URLs to HTTPS format")
print(f"First URL: {https_urls[0]}")

## Step 3: Set up Remote File Access

Configure object store and parser for accessing remote Copernicus files.

In [ ]:
# Create object-store handle for remote files
url_prefix = f"{COPERNICUS_ENDPOINT}/"
store = from_url(url_prefix)
registry = ObjectStoreRegistry({url_prefix: store})

# Use HDF parser for NetCDF files
parser = HDFParser()

print(f"✓ Remote storage configured for: {url_prefix}")

## Step 4: Create Icechunk Repository

Set up local Icechunk storage with virtual chunk configuration.

In [ ]:
# Set up local storage path
repo_path = Path("./copernicus_icechunk_append")
if repo_path.exists():
    shutil.rmtree(repo_path)
    print(f"Cleared existing repo at {repo_path}/")

# Configure virtual chunk container
# This tells Icechunk where the actual data chunks live
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=url_prefix,
        store=icechunk.http_store(),
    )
)

# Create local repository
storage = icechunk.local_filesystem_storage(str(repo_path))
repo = icechunk.Repository.create(storage, config)

# Create writable session
session = repo.writable_session("main")

print(f"✓ Created Icechunk repository at {repo_path}")

## Step 5: Loop Through Files and Append

For each file:
1. Open as virtual dataset (no download)
2. First file: write to Icechunk
3. Subsequent files: append along time dimension

In [ ]:
# Process all files
for i, url in enumerate(https_urls):
    filename = Path(url).name
    start = time.perf_counter()
    
    print(f"[{i+1}/{len(https_urls)}] Adding {filename}...")
    
    # Open file virtually (no data download)
    vds = open_virtual_dataset(
        url=url,
        parser=parser,
        registry=registry,
        loadable_variables=['time', 'lat', 'lon', 'latitude', 'longitude'],
        indexes={},
    )
    
    # First file: create initial dataset
    # Subsequent files: append along time dimension
    if i == 0:
        vds.virtualize.to_icechunk(session.store)
    else:
        vds.virtualize.to_icechunk(session.store, append_dim="time")
    
    elapsed = time.perf_counter() - start
    print(f"  ✓ Finished in {elapsed:.2f} seconds")

print("\n✓ All files processed")

## Step 6: Commit Changes

Commit all virtual references to the Icechunk repository.

In [ ]:
# Commit all changes
snapshot_id = session.commit(f"Added {len(https_urls)} days of Copernicus chlorophyll data")
print(f"✓ Committed snapshot: {snapshot_id}")

## Step 7: Read and Verify

Open the Icechunk store and verify we have all time steps.

In [ ]:
# Open repository for reading
repo_read = icechunk.Repository.open(storage, config=config)
session_read = repo_read.readonly_session(branch="main")

# Open with xarray
ds = xr.open_zarr(session_read.store, consolidated=False)

print("✓ Dataset opened from Icechunk store:")
print(ds)
print(f"\nTime dimension has {len(ds.time)} steps")

## Summary

This notebook demonstrated:

1. ✓ Getting S3 URLs for multiple Copernicus files (no download)
2. ✓ Converting to HTTPS format
3. ✓ Creating Icechunk repository with virtual chunk configuration
4. ✓ Looping through files and appending along time dimension
5. ✓ Committing all virtual references
6. ✓ Reading back the combined dataset

**Key Achievement**: Created a time series dataset from multiple Copernicus files with only metadata and virtual references. No data was downloaded or duplicated.

### Comparison with NDVI Example

| Aspect | NDVI CDR | Copernicus Marine |
|--------|----------|-------------------|
| **Data Source** | NOAA CDR public S3 | Copernicus cloudferro S3 |
| **Access** | Anonymous | Requires Copernicus credentials |
| **URL Format** | `s3://noaa-cdr-ndvi-pds/` | `https://s3.waw3-1.cloudferro.com/` |
| **Store Config** | `icechunk.s3_store(anonymous=True)` | `icechunk.http_store()` |